<a href="https://colab.research.google.com/github/charles-turner-1/random-python-ideas/blob/main/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
sys.version

'3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]'

In [2]:
# Import basic libraries for file handling and data manipulation
import os
import pandas as pd

# Login to Kaggle Hub - get credentials from https://www.kaggle.com/settings
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
path = kagglehub.model_download("keras/gemma3/keras/gemma3_instruct_270m")


  0%|          | 0.00/3.23k [00:00<?, ?B/s]

100%|██████████| 3.23k/3.23k [00:00<00:00, 403kB/s]


  0%|          | 0.00/184 [00:00<?, ?B/s]

100%|██████████| 184/184 [00:00<00:00, 222kB/s]

100%|██████████| 965/965 [00:00<00:00, 2.41MB/s]



  0%|          | 0.00/512M [00:00<?, ?B/s]



100%|██████████| 596/596 [00:00<00:00, 1.28MB/s]




100%|██████████| 1.47k/1.47k [00:00<00:00, 3.61MB/s]




  0%|          | 0.00/4.47M [00:00<?, ?B/s]
  0%|          | 1.00M/512M [00:00<01:38, 5.45MB/s]

 22%|██▏       | 1.00M/4.47M [00:00<00:00, 5.25MB/s]
100%|██████████| 4.47M/4.47M [00:00<00:00, 17.3MB/s]

  4%|▍         | 20.0M/512M [00:00<00:07, 67.0MB/s]
  6%|▌         | 32.0M/512M [00:00<00:05, 86.3MB/s]
  9%|▊         | 44.0M/512M [00:00<00:05, 98.1MB/s]
 11%|█         | 56.0M/512M [00:00<00:04, 104MB/s] 
 13%|█▎        | 68.0M/512M [00:00<00:04, 110MB/s]
 16%|█▌        | 80.0M/512M [00:00<00:03, 113MB/s]
 18%|█▊        | 92.0M/512M [00:01<00:03, 115MB/s]
 20%|██        | 104M/512M [00:01<00:03, 118MB/s] 
 23%|██▎       | 116M/512M [00:01<00:03, 117MB/s]
 25%|██▍       | 128M/512M [00:01<00:03, 119MB/s]
 27%|██▋       | 140M/512M [00:01<00:03, 119MB/s]
 30%|██▉       | 152M/512M [00:01<00:03, 120MB/s]
 32%|███▏      | 164M/512M [00:01<00:03, 120MB/s]
 34%|███▍      | 176M/512M [00:01<00:02, 119MB/s]
 37%|███▋      | 188M/512M [00:01<00:02, 120MB/s]
 39%|███▉      | 200M/512M [00:

In [ ]:
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"


In [6]:
! pip install tensorflow keras_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 19.7 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


In [7]:
# --- Import Deep Learning Libraries ---

# Import JAX and configure it to use the TPU.
import jax
jax.config.update('jax_platform_name', 'tpu')
print(f"JAX is running on {jax.devices()[0].device_kind}")

# Import our main deep learning frameworks: Keras and keras-hub (forerly keras-nlp) for LLM-specific tools.
import keras
import keras_hub

# bfloat16 uses less memory than the standard float32, which helps our model train faster on a TPU without a major loss in accuracy.
# keras.config.set_floatx("bfloat16")

JAX is running on TPU v5 lite


In [11]:
path_data = kagglehub.dataset_download("gpreda/medquad")
path_data

Using Colab cache for faster access to the 'medquad' dataset.


'/kaggle/input/medquad'

In [10]:
# Load and subset the data for training
df = pd.read_csv(path_data+"/medquad.csv")
# data = df.sample(n=100, random_state=42)

# For this workshop, we want the fine-tuning process to be fast and the results to be obvious.
# So, we will "cheat" by creating a very small, highly specific dataset focused only on "pernicious anemia".
# In a real-world project, you would use a much larger and more diverse dataset representing your entire domain.
df_subset_mask = df['question'].str.contains('pernicious anemia', case=False, na=False) | \
                         df['answer'].str.contains('pernicious anemia', case=False, na=False) | \
                         df['focus_area'].str.contains('pernicious anemia', case=False, na=False)
df_subset = df[df_subset_mask]

# Preview the first few lines of the data
df_subset.head(2)

Using Colab cache for faster access to the 'medquad' dataset.


,question,answer,source,focus_area
1132,Who is at risk for Gastrointestinal Carcinoid ...,Health history can affect the risk of gastroin...,CancerGov,Gastrointestinal Carcinoid Tumors
3017,What is (are) Autoimmune atrophic gastritis ?,Autoimmune atrophic gastritis is an autoimmune...,GARD,Autoimmune atrophic gastritis


In [13]:
# Helper function to transform our dataframe into the required format.
def format_data(df):
    prompts = []
    responses = []
    for index, row in df.iterrows():
        question = row['question']
        response = row['answer']
        if question and response:
             # prompts.append(f"<start_of_turn>user\nInstruction:\nAnswer the following question.\nQuestion:{question}\n<end_of_turn>")
             # responses.append(f"<start_of_turn>model\nResponse:{response}\n<end_of_turn>")
            prompts.append(f"{question}")
            responses.append(f"{response}")

    data_to_preprocess = {"prompts": prompts, "responses": responses}
    return data_to_preprocess

# Apply the formatting to our data.
formatted_data = format_data(df_subset)
type

dict

In [17]:
# From the kagglehub download output (cell 3)
path_gemma = "/root/.cache/kagglehub/models/keras/gemma3/keras/gemma3_instruct_270m/4"

In [18]:
# Load the Gemma3 model
# `from_preset` is a convenient Keras function to load a model with its standard configuration.
# This includes the model architecture itself, the pre-trained weights, and the tokenizer
# which converts text into numbers the model can understand.
# We are loading a smaller 270 Million parameter version of Gemma 3, which is suitable for quick fine-tuning.
print("Loading model...")
causal_lm = keras_hub.models.Gemma3CausalLM.from_preset(path_gemma)

# The .summary() method gives us a look at the model's architecture.
# Pay attention to the "Total params" and "Trainable params". In this full fine-tuning
# example, they will be the same, meaning we are updating every part of the model.
causal_lm.summary()

Loading model...


Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 640)         │     268,098,176 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     167,772,160 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 268,098,176 (1022.71 MB)

 Trainable params: 268,098,176 (1022.71 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
prompt = "What is pernicious anemia?"
response_raw = causal_lm.generate(prompt)

print(f"{response_raw}")

What is pernicious anemia?

The answer is that it is a condition where the body's ability to produce enough red blood cells is impaired. This can lead to a variety of symptoms, including fatigue, weakness, shortness of breath, and dizziness.

What is pernicious anemia?

The answer is that it is a condition where the body's ability to produce enough red blood cells is impaired. This can lead to a variety of symptoms, including fatigue, weakness, shortness of breath, and dizziness.

What is the main cause of pernicious anemia?

The answer is that it is a condition where the body's ability to produce enough red blood cells is impaired. This can lead to a variety of symptoms, including fatigue, weakness, shortness of breath, and dizziness.

What is the main symptom of pernicious anemia?

The answer is that it is a condition where the body's ability to produce enough red blood cells is impaired. This can lead to a variety of symptoms, including fatigue, weakness, shortness of breath, and di

In [22]:
# Enable Low-Rank Adaptation (LoRA) for parameter efficient fine-tuning.
# LoRA freezes all weights on the backbone except for specific attention layer components
causal_lm.backbone.enable_lora(rank=16)
print(f"Number of trainable weights after LoRA: {len(causal_lm.trainable_weights)}")
print(f"Number of non-trainable weights after LoRA: {len(causal_lm.non_trainable_weights)}")

Number of trainable weights after LoRA: 72
Number of non-trainable weights after LoRA: 236


In [23]:
print("Compiling the model...")
causal_lm.compile(
    # The optimizer is the algorithm that updates the model's weights to minimize the loss.
    # Adam is a very popular and effective general-purpose optimizer.
    # The `learning_rate` is the single most important hyperparameter. It controls the size of the
    # weight updates. Too large, and the training can become unstable; too small, and it will be too slow.
    # A small learning rate like 1e-4 (0.0001) is a good starting point for fine-tuning.
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    # The "loss function" calculates a score that measures how wrong the model's predictions are.
    # The goal of training is to minimize this score. SparseCategoricalCrossentropy is the standard
    # loss function for next-token prediction tasks.
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    # Metrics are used to monitor the training process. Here, we'll track accuracy.
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()]
)
print("Done.")

Compiling the model...
Done.


In [24]:
print("Starting fine-tuning...")
causal_lm.fit(formatted_data, epochs=10, batch_size=1) # Adjust batch_size depening on VRAM available. Adjust epoch until loss plateaus
print("Fine-tuning complete!")

Starting fine-tuning...
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 281s 12s/step - loss: 1.0651 - sparse_categorical_accuracy: 0.5171
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 258s 12s/step - loss: 0.9848 - sparse_categorical_accuracy: 0.5291
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 258s 12s/step - loss: 0.9507 - sparse_categorical_accuracy: 0.5402
Epoch 4/10
 8/21 ━━━━━━━━━━━━━━━━━━━━ 2:39 12s/step - loss: 0.4454 - sparse_categorical_accuracy: 0.5369

KeyboardInterrupt: 

In [ ]:
print("Testing generation from the fine-tuned model:")
response_ft = causal_lm.generate(prompt)
print(f"{response_ft}")